# PKG community views: clearer pictures of the same communities

Companion to `pkg_community_map_workshop.ipynb`. It reads that notebook's outputs from `OUT_DIR` (edges, nodes, membership, hub list), so run that one first. Nothing here needs Spark.

**What is different from the first notebook**

| First notebook | Problem on real data | Here |
|---|---|---|
| Sector mix bars | the 7 sectors largest overall are kept and the rest is one grey bar, so every community looks like the average | **Sector lift heatmap**: how over- or under-represented each sector is in each community |
| Community map | every community trades a little with every other one, so the layout collapses into the centre | **Map of corridors**: only links much stronger than the two communities' sizes predict, every bubble labelled with what the community is |
| (none) | | **Flow matrix**: community × community, ordered so trading partners sit together; nothing can overlap |
| Drill-down | the top members by dollars are not connected to each other, so the layout pushes stragglers to the edge | **Drill-down by sub-community**: a connected snowball from the strongest member, coloured by sub-community |

Sector colours are fixed for the whole notebook: the same sector always has the same colour.

**Figures in Jupyter.** `RENDERER = "iframe"` draws every figure in the notebook from a small HTML file with plotly.js inside, so it works without internet and without the JupyterLab plotly extension. If figures are still blank, try `"iframe_connected"` (loads plotly.js from the internet) or `"png"` (static pictures, needs the `kaleido` package). Every figure is also saved as an HTML file in `OUT_DIR` that opens in a browser.

**Governance.** Real customer data: present from your own screen and don't share the files. `SHOW_NAMES = False` keeps names out of every picture.

In [ ]:
# Config
OUT_DIR             = "../metrics/community/workshop_map"   # same folder as the map notebook
MIN_PAIR_AMOUNT     = 0.0     # same value as the map notebook
TOP_K_COMMUNITIES   = 40      # communities in the map and the heatmaps (largest first)
MIN_COMMUNITY_SIZE  = 10
MIN_SECTOR_SHARE    = 0.01    # heatmap columns: sectors with at least this share of all members
LINK_LIFT_MIN       = 2.0     # map: keep a link when its dollars are at least 2x what sizes predict ...
LINKS_PER_COMMUNITY = 2       # ... and it is among each community's 2 strongest such links
DRILL_COMMUNITIES   = None    # None = the two largest; or a list such as ["C003", "C011"]
DRILL_NODES         = 300     # nodes drawn in a drill-down (connected snowball from the strongest member)
RENDERER            = "iframe"   # "iframe" (offline), "iframe_connected", or "png" (needs kaleido)
SHOW_NAMES          = False
SEED                = 42

In [ ]:
# Setup: imports, fixed sector colours, one-output-per-cell display helper
import os, math, time, base64, warnings
warnings.filterwarnings("ignore")
import numpy as np
for _alias, _target in (("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)):
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

pio.renderers.default = RENDERER

NAICS2 = {"11": "Agriculture", "21": "Mining", "22": "Utilities", "23": "Construction",
          "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing", "42": "Wholesale",
          "44": "Retail", "45": "Retail", "48": "Transportation", "49": "Transportation",
          "51": "Information", "52": "Finance and insurance", "53": "Real estate",
          "54": "Professional services", "55": "Holding companies", "56": "Admin and support",
          "61": "Education", "62": "Health care", "71": "Arts and recreation",
          "72": "Food services", "81": "Other services", "92": "Public administration"}
SECTORS = list(dict.fromkeys(NAICS2.values()))
_COLS = ["#7a9a01", "#a8641e", "#14a3b8", "#eb6834", "#2a78d6", "#1baf7a", "#eda100", "#6250d6",
         "#3d5a80", "#e87ba4", "#b05cc6", "#5c946e", "#8d6e63", "#f2a65a", "#00897b", "#e34948",
         "#9e9d24", "#c2185b", "#008300", "#5e35b1"]
SECTOR_COLOR = {s: _COLS[i % len(_COLS)] for i, s in enumerate(SECTORS)}
UNKNOWN = "#d6d5cf"
SURFACE, INK, MUTED, EDGE = "#fcfcfb", "#0b0b0b", "#52514e", "#c9c8c0"


def usd(x):
    x = float(x)
    for unit, d in (("B", 1e9), ("M", 1e6), ("K", 1e3)):
        if abs(x) >= d:
            return f"USD {x / d:,.1f}{unit}"
    return f"USD {x:,.0f}"


_FIG_N = [0]


def fig_html(fig):
    """A figure as HTML the notebook can show. Each figure gets its own small HTML file in
    iframe_figures/ (next to this notebook) with plotly.js inside, shown in an iframe."""
    if RENDERER == "png":
        img = fig.to_image(format="png", width=int(fig.layout.width or 1400),
                           height=int(fig.layout.height or 800))
        return f"<img style='max-width:100%' src='data:image/png;base64,{base64.b64encode(img).decode()}'>"
    _FIG_N[0] += 1
    os.makedirs("iframe_figures", exist_ok=True)
    path = f"iframe_figures/views_{_FIG_N[0]:02d}.html"
    fig.write_html(path, include_plotlyjs=True if RENDERER == "iframe" else "cdn", default_width="100%")
    h = int(fig.layout.height or 525) + 20
    return f'<iframe src="{path}" width="100%" height="{h}" frameborder="0" scrolling="no"></iframe>'


def show(title, items):
    """One consolidated output per cell: text, tables and figures in one HTML block."""
    parts = [f"<h4 style='margin:6px 0'>{title}</h4>"]
    for it in items:
        if isinstance(it, pd.DataFrame):
            parts.append(it.to_html(index=False, border=0, justify="left",
                                    float_format=lambda x: f"{x:,.2f}"))
        elif isinstance(it, go.Figure):
            parts.append(fig_html(it))
        else:
            parts.append(f"<div style='margin:2px 0'>{it}</div>")
    display(HTML("".join(parts)))


def save(fig, name):
    path = os.path.join(OUT_DIR, f"{name}.html")
    fig.write_html(path, include_plotlyjs=True)
    return path


def style(fig, title, height):
    fig.update_layout(title=dict(text=title, x=0.01, font=dict(size=16, color=INK)), height=height,
                      plot_bgcolor=SURFACE, paper_bgcolor=SURFACE, font=dict(color=INK),
                      margin=dict(l=10, r=10, t=60, b=10))
    return fig


show("Setup", [f"pandas {pd.__version__} &middot; networkx {nx.__version__} &middot; plotly "
               f"{__import__('plotly').__version__} &middot; renderer <b>{RENDERER}</b>",
               f"OUT_DIR {os.path.abspath(OUT_DIR)}"])

## 1. Load the map notebook's outputs

Rebuilds the same de-hubbed customer-pair graph and attaches each member's community, sector and dollars.

In [ ]:
# V1. Load edges, nodes, membership and hubs; rebuild the de-hubbed pair table
t0 = time.time()


def _read(name):
    p = os.path.join(OUT_DIR, name)
    if os.path.exists(p + ".parquet"):
        return pd.read_parquet(p + ".parquet")
    return pd.read_csv(p + ".csv", dtype={"src": str, "dst": str, "node": str})


edges = _read("edges")
nodes = _read("nodes")
nodes["node"] = nodes.node.astype(str)
nodes = nodes.set_index("node")
nodes["sector"] = [NAICS2.get(str(c)[:2]) if s == "valid" and pd.notna(c) else None
                   for c, s in zip(nodes.naics2, nodes.naics_status)]
membership = _read("membership")
membership["node"] = membership.node.astype(str)
comm = membership.set_index("node").community
hubs = set(pd.read_csv(os.path.join(OUT_DIR, "hubs_removed.csv"), dtype={"node": str}).node)

e = edges[edges.amount > MIN_PAIR_AMOUNT]
a, b = e.src.astype(str).to_numpy(dtype=object), e.dst.astype(str).to_numpy(dtype=object)
pairs = (pd.DataFrame({"u": np.where(a < b, a, b), "v": np.where(a < b, b, a), "amount": e.amount.to_numpy()})
         .groupby(["u", "v"], as_index=False).amount.sum())
pairs = pairs[~pairs.u.isin(hubs) & ~pairs.v.isin(hubs)].reset_index(drop=True)
pairs["cu"], pairs["cv"] = pairs.u.map(comm), pairs.v.map(comm)
pairs = pairs.dropna(subset=["cu", "cv"])

ends = pd.concat([pairs[["u", "amount"]].rename(columns={"u": "node"}),
                  pairs[["v", "amount"]].rename(columns={"v": "node"})])
node_df = pd.DataFrame({"degree": ends.groupby("node").size(), "strength": ends.groupby("node").amount.sum()})
node_df["community"] = comm.reindex(node_df.index)
node_df["sector"] = nodes.sector.reindex(node_df.index)

sizes = node_df.community.value_counts()
top_ids = sizes[sizes >= MIN_COMMUNITY_SIZE].index[:TOP_K_COMMUNITIES].tolist()
same = pairs.cu == pairs.cv
show("V1. Loaded", [
    f"{len(node_df):,} businesses in {node_df.community.nunique():,} communities &middot; "
    f"{len(pairs):,} customer pairs &middot; {usd(pairs.amount.sum())} &middot; {len(hubs):,} hubs left out",
    f"{same.mean():.0%} of pairs and {pairs.amount[same].sum() / pairs.amount.sum():.0%} of dollars stay inside a community",
    f"Members with a valid NAICS sector: {node_df.sector.notna().mean():.0%}",
    f"Pictures use the {len(top_ids)} largest communities (at least {MIN_COMMUNITY_SIZE} members) &middot; "
    f"{time.time() - t0:,.0f}s",
])

## 2. What is each community? Sector lift

Lift = a sector's share of a community's members divided by its share of all members. Lift 3 means the sector is three times as common here as in the network overall; lift 1 means average. Lift shows what makes a community different, which the plain mix hides when a few large sectors are everywhere.

Each community gets a short name from its two highest-lift sectors that cover at least 10% of its members. Members without a valid NAICS are left out of the shares and reported separately.

In [ ]:
# V2. Sector lift per community and a short name for each community
known = node_df.dropna(subset=["sector"])
cnt = known.groupby(["community", "sector"]).size().unstack(fill_value=0)
share = cnt.div(cnt.sum(1), axis=0)
overall = cnt.sum() / cnt.sum().sum()
lift = share / overall

names = {}
for c in sizes.index:
    if c not in share.index:
        names[c] = "sector unknown"
        continue
    cand = lift.loc[c][share.loc[c] >= 0.10].sort_values(ascending=False)
    names[c] = " + ".join(cand.index[:2]) if len(cand) else share.loc[c].idxmax()
signature = {c: (n.split(" + ")[0] if n != "sector unknown" else None) for c, n in names.items()}

inside = pairs[same].groupby("cu").amount.sum()
touch = pd.concat([pairs.groupby("cu").amount.sum(), pairs[~same].groupby("cv").amount.sum()]).groupby(level=0).sum()
prof = pd.DataFrame({"community": top_ids})
prof["members"] = prof.community.map(sizes)
prof["unknown_sector"] = prof.community.map(lambda c: 1 - known.community.eq(c).sum() / sizes[c])
prof["name"] = prof.community.map(names)
prof["dollars_inside"] = prof.community.map(inside).fillna(0)
prof["inside_share"] = prof.dollars_inside / prof.community.map(touch)
prof["top_by_lift"] = [", ".join(f"{s} {share.loc[c, s]:.0%} (x{lift.loc[c, s]:.1f})"
                                 for s in lift.loc[c][share.loc[c] >= 0.05].sort_values(ascending=False).index[:3])
                       if c in share.index else "" for c in prof.community]
prof.to_csv(os.path.join(OUT_DIR, "community_names.csv"), index=False)

view = prof.head(20).copy()
view["dollars_inside"] = view.dollars_inside.map(usd)
view["inside_share"] = view.inside_share.map("{:.0%}".format)
view["unknown_sector"] = view.unknown_sector.map("{:.0%}".format)
show("V2. The twenty largest communities, named by sector lift", [
    view[["community", "members", "name", "top_by_lift", "inside_share", "dollars_inside", "unknown_sector"]],
    "top_by_lift: share of members (lift against the whole network). inside_share: of the dollars touching "
    "the community, the share that stays inside. Saved community_names.csv.",
])

In [ ]:
# V3. Heatmap: sector lift for each community (orange = over-represented, blue = under)
cols = [s for s in overall.sort_values(ascending=False).index if overall[s] >= MIN_SECTOR_SHARE]
rows = [c for c in top_ids if c in share.index]
rows = sorted(rows, key=lambda c: (cols.index(signature[c]) if signature[c] in cols else 99, -sizes[c]))
L = np.log2(lift.loc[rows, cols].replace(0, np.nan)).clip(-3, 3).fillna(-3)
S = share.loc[rows, cols]
txt = S.applymap(lambda x: f"{x:.0%}" if x >= 0.05 else "") if hasattr(S, "applymap") else S.map(lambda x: f"{x:.0%}" if x >= 0.05 else "")
hover = [[f"{r} ({names[r]})<br>{s}: {S.loc[r, s]:.1%} of members<br>network: {overall[s]:.1%}"
          f"<br>lift x{lift.loc[r, s]:.1f}" for s in cols] for r in rows]
fig = go.Figure(go.Heatmap(
    z=L.to_numpy(), x=cols, y=[f"{r}  {names[r][:44]}" for r in rows], text=txt.to_numpy(),
    texttemplate="%{text}", textfont=dict(size=10), hovertext=hover, hoverinfo="text",
    zmin=-3, zmax=3, colorscale=[[0, "#a9c8ee"], [0.5, "#f7f6f2"], [1, "#c94f1e"]],
    colorbar=dict(title="lift", tickvals=[-3, -2, -1, 0, 1, 2, 3],
                  ticktext=["1/8", "1/4", "1/2", "1", "2", "4", "8"])))
style(fig, "Sector lift by community (cell text = share of members, shown from 5%)", 260 + 24 * len(rows))
fig.update_layout(xaxis=dict(side="top", tickangle=-40), yaxis=dict(autorange="reversed"),
                  margin=dict(l=10, r=10, t=170, b=10))
show("V3. Sector lift heatmap", [
    fig,
    f"{len(rows)} communities x {len(cols)} sectors. Rows are grouped by each community's highest-lift sector. "
    "Read across a row to see what a community is made of; down a column to see where a sector concentrates.",
    "Saved " + save(fig, "sector_lift_heatmap"),
])

## 3. Between communities: corridors, not a hairball

Two large communities exchange a lot of dollars simply because they are large. Lift corrects for that: observed dollars between A and B divided by what their total dollars would predict if money went anywhere at random. The map keeps a link only when its lift is at least `LINK_LIFT_MIN` and it is among the community's `LINKS_PER_COMMUNITY` strongest; each community also keeps its single largest link by dollars so nothing floats away.

In [ ]:
# V4. Corridor map: one labelled bubble per community, links only where lift is high
t0 = time.time()
cross = pairs[~same & pairs.cu.isin(top_ids) & pairs.cv.isin(top_ids)]
lk = cross.assign(a=np.minimum(cross.cu, cross.cv), b=np.maximum(cross.cu, cross.cv)).groupby(["a", "b"]).amount.sum()
lk = lk.reset_index()
vol = pd.concat([pairs.groupby("cu").amount.sum(), pairs.groupby("cv").amount.sum()]).groupby(level=0).sum()
W = pairs.amount.sum()
lk["expected"] = lk.a.map(vol) * lk.b.map(vol) / (2 * W)
lk["lift"] = lk.amount / lk.expected
cand = lk[lk.lift >= LINK_LIFT_MIN].copy()
cand["ra"] = cand.groupby("a").lift.rank(ascending=False)
cand["rb"] = cand.groupby("b").lift.rank(ascending=False)
keep = cand[(cand.ra <= LINKS_PER_COMMUNITY) | (cand.rb <= LINKS_PER_COMMUNITY)]
both = pd.concat([lk.rename(columns={"a": "x", "b": "y"}), lk.rename(columns={"b": "x", "a": "y"})])
biggest = both.loc[both.groupby("x").amount.idxmax()][["x", "y", "amount", "lift"]]
biggest = biggest.assign(a=np.minimum(biggest.x, biggest.y), b=np.maximum(biggest.x, biggest.y))[["a", "b", "amount", "lift"]]
links = pd.concat([keep[["a", "b", "amount", "lift"]], biggest]).drop_duplicates(["a", "b"]).reset_index(drop=True)

QG = nx.Graph()
QG.add_nodes_from(top_ids)
for r in links.itertuples():
    QG.add_edge(r.a, r.b, weight=float(np.log2(max(r.lift, 1.0001))))
pos = nx.spring_layout(QG, weight="weight", seed=SEED, k=3.0 / math.sqrt(len(QG)), iterations=400)

lw = np.log2(links.lift.clip(lower=1))
traces = []
for lo, hi, wd in ((0, 2, 1.2), (2, 3, 2.6), (3, 99, 4.5)):
    m = (lw >= lo) & (lw < hi)
    xs, ys = [], []
    for r in links[m].itertuples():
        xs += [pos[r.a][0], pos[r.b][0], None]
        ys += [pos[r.a][1], pos[r.b][1], None]
    traces.append(go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=wd, color=EDGE),
                             hoverinfo="skip", showlegend=False))
# invisible midpoints carry the link hover
traces.append(go.Scatter(x=[(pos[r.a][0] + pos[r.b][0]) / 2 for r in links.itertuples()],
                         y=[(pos[r.a][1] + pos[r.b][1]) / 2 for r in links.itertuples()], mode="markers",
                         marker=dict(size=10, color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="text",
                         hovertext=[f"{r.a} - {r.b}<br>{usd(r.amount)}<br>lift x{r.lift:.1f}" for r in links.itertuples()]))
p = prof.set_index("community")
for sec in [s for s in SECTORS if s in set(signature[c] for c in top_ids)] + [None]:
    ids = [c for c in top_ids if signature[c] == sec]
    if not ids:
        continue
    traces.append(go.Scatter(
        x=[pos[c][0] for c in ids], y=[pos[c][1] for c in ids], mode="markers+text", name=sec or "sector unknown",
        marker=dict(size=[14 + 46 * math.sqrt(sizes[c] / sizes[top_ids[0]]) for c in ids],
                    color=SECTOR_COLOR.get(sec, UNKNOWN), line=dict(width=2, color=SURFACE)),
        text=[f"<b>{c}</b><br>{names[c][:28]}" for c in ids], textposition="bottom center",
        textfont=dict(size=10, color=INK),
        hovertext=[f"<b>{c}</b>: {names[c]}<br>{sizes[c]:,} members<br>{p.top_by_lift[c]}<br>"
                   f"dollars inside {usd(p.dollars_inside[c])} ({p.inside_share[c]:.0%})" for c in ids],
        hoverinfo="text"))
fig = go.Figure(traces)
style(fig, f"Community corridors: {len(top_ids)} largest communities, {len(links)} links "
           f"(colour = highest-lift sector, link width = lift)", 900)
fig.update_layout(xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
                  legend=dict(orientation="h", y=-0.02))
strong = links.sort_values("lift", ascending=False).head(8)
tab = pd.DataFrame({"from": strong.a, "to": strong.b, "names": [f"{names[x]}  /  {names[y]}" for x, y in zip(strong.a, strong.b)],
                    "dollars": strong.amount.map(usd), "lift": strong.lift.round(1)})
show("V4. Corridor map", [fig, "Strongest corridors (dollars far above what the two sizes predict):", tab,
                          f"Saved {save(fig, 'community_corridors')} &middot; {time.time() - t0:,.0f}s"])

In [ ]:
# V5. Flow matrix: community x community lift, ordered so trading partners sit together
from scipy.cluster.hierarchy import linkage, leaves_list
M = pd.DataFrame(1.0, index=top_ids, columns=top_ids)
for r in lk.itertuples():
    M.loc[r.a, r.b] = M.loc[r.b, r.a] = r.lift
for c in top_ids:
    M.loc[c, c] = inside.get(c, 0) / max(vol[c] ** 2 / (2 * W), 1e-9)
Z = np.log2(M.clip(lower=1 / 8, upper=64))
order = [top_ids[i] for i in leaves_list(linkage(Z.to_numpy(), "average"))]
Z = Z.loc[order, order]
lab = [f"{c} {names[c][:32]}" for c in order]
fig = go.Figure(go.Heatmap(
    z=Z.to_numpy(), x=lab, y=lab, zmin=-3, zmax=6,
    colorscale=[[0, "#a9c8ee"], [1 / 3, "#f7f6f2"], [1, "#c94f1e"]],
    hovertext=[[f"{a} - {b}<br>lift x{M.loc[a, b]:.1f}" for b in order] for a in order], hoverinfo="text",
    colorbar=dict(title="lift", tickvals=[-3, 0, 2, 4, 6], ticktext=["1/8", "1", "4", "16", "64"])))
style(fig, "Dollar flows between communities, relative to what their sizes predict "
           "(diagonal = inside the community)", 300 + 22 * len(order))
fig.update_layout(xaxis=dict(tickangle=-50, tickfont=dict(size=9)), yaxis=dict(autorange="reversed", tickfont=dict(size=9)))
show("V5. Flow matrix", [fig, "Orange blocks off the diagonal are groups of communities that trade heavily with "
                         "each other: supply-chain corridors. Blue = they trade less than their sizes predict.",
                         "Saved " + save(fig, "community_flow_matrix")])

## 4. Inside one community

The drawing starts at the community's strongest member and grows outward along its largest payment links until `DRILL_NODES` members, so everything drawn is connected. Colour = sub-community, found by running Louvain again inside the whole community; the bar chart names each sub-community by its highest-lift sector against the community.

In [ ]:
# V6. Drill-down: connected snowball inside a community, coloured by sub-community
t0 = time.time()
drill = DRILL_COMMUNITIES or top_ids[:2]
SUB_COLS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
items = []
for c in drill:
    pc = pairs[(pairs.cu == c) & (pairs.cv == c)]
    G = nx.Graph()
    G.add_weighted_edges_from(zip(pc.u, pc.v, np.log1p(pc.amount)))
    for x, y, amt_ in zip(pc.u, pc.v, pc.amount):
        G[x][y]["amount"] = amt_
    try:
        import igraph as ig, leidenalg
        g = ig.Graph.TupleList(G.edges(data="weight"), weights=True)
        part = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition, weights="weight", seed=SEED)
        sub = {g.vs[i]["name"]: k for i, k in enumerate(part.membership)}
    except Exception:
        sub = {v: k for k, cm in enumerate(nx.community.louvain_communities(G, weight="weight", seed=SEED)) for v in cm}
    sub_sizes = pd.Series(sub).value_counts()
    rank = {k: i for i, k in enumerate(sub_sizes.index)}
    sub = {v: rank[k] for v, k in sub.items()}

    # snowball: strongest member first, then neighbours by largest link
    start = node_df.loc[list(G)].strength.idxmax()
    seen, frontier = {start}, [start]
    while frontier and len(seen) < DRILL_NODES:
        nxt = []
        for x in frontier:
            for y in sorted(G[x], key=lambda y: -G[x][y]["amount"]):
                if y not in seen and len(seen) < DRILL_NODES:
                    seen.add(y)
                    nxt.append(y)
        frontier = nxt
    H = G.subgraph(seen).copy()
    p2 = nx.spring_layout(H, weight="weight", seed=SEED, k=1.8 / math.sqrt(len(H)), iterations=200)
    pr = nx.pagerank(H, weight="weight")
    alias = {x: (nodes.customer_name.get(x) if SHOW_NAMES else f"N{i + 1:03d}") for i, x in enumerate(H)}

    # name the sub-communities by lift against the community
    sec_c = node_df.loc[list(G)].sector
    base = sec_c.value_counts(normalize=True)
    sub_lab = {}
    for k in range(min(len(SUB_COLS), len(sub_sizes))):
        s = sec_c[[v for v in G if sub[v] == k]].value_counts(normalize=True)
        lifted = (s / base.reindex(s.index))[s >= 0.15].sort_values(ascending=False)
        sub_lab[k] = lifted.index[0] if len(lifted) else (s.index[0] if len(s) else "sector unknown")

    fig = make_subplots(rows=1, cols=2, column_widths=[0.7, 0.3], horizontal_spacing=0.04,
                        subplot_titles=(f"{len(H)} connected members around the strongest one",
                                        "Sub-communities (whole community)"))
    xs, ys = [], []
    for x, y in H.edges():
        xs += [p2[x][0], p2[y][0], None]
        ys += [p2[x][1], p2[y][1], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=0.6, color=EDGE), hoverinfo="skip",
                             showlegend=False), 1, 1)
    mx = max(pr.values())
    for k in list(range(len(sub_lab))) + [None]:
        nl = [x for x in H if (sub[x] == k if k is not None else sub[x] >= len(sub_lab))]
        if not nl:
            continue
        fig.add_trace(go.Scatter(
            x=[p2[x][0] for x in nl], y=[p2[x][1] for x in nl], mode="markers",
            name=f"S{k + 1} {sub_lab[k]}" if k is not None else "smaller sub-communities",
            marker=dict(size=[6 + 22 * math.sqrt(pr[x] / mx) for x in nl],
                        color=SUB_COLS[k] if k is not None else UNKNOWN, line=dict(width=1, color=SURFACE)),
            hovertext=[f"<b>{alias[x]}</b><br>{node_df.sector.get(x) or 'sector unknown'}<br>"
                       f"links in this view {H.degree(x)}<br>{usd(node_df.strength[x])} with other businesses"
                       for x in nl], hoverinfo="text"), 1, 1)
    ks = list(sub_lab)[::-1]
    fig.add_trace(go.Bar(y=[f"S{k + 1} {sub_lab[k]}" for k in ks], x=[sub_sizes.iloc[k] for k in ks], orientation="h",
                         marker=dict(color=[SUB_COLS[k] for k in ks]), showlegend=False,
                         hovertemplate="%{y}: %{x:,} members<extra></extra>"), 1, 2)
    style(fig, f"{c}: {names[c]} · {G.number_of_nodes():,} members, {len(sub_sizes):,} sub-communities "
               f"(names masked)", 760)
    fig.update_xaxes(visible=False, row=1, col=1)
    fig.update_yaxes(visible=False, scaleanchor="x", row=1, col=1)
    fig.update_layout(legend=dict(orientation="h", y=-0.04))
    items += [fig, "Saved " + save(fig, f"drill_v2_{c}")]
show("V6. Drill-downs", items + [f"{time.time() - t0:,.0f}s"])

In [ ]:
# V7. Files written by this notebook
new = [f for f in sorted(os.listdir(OUT_DIR)) if f.startswith(("sector_lift", "community_corridors",
                                                                  "community_flow", "drill_v2", "community_names"))]
show("V7. Outputs", [pd.DataFrame({"file": new, "size_kb": [round(os.path.getsize(os.path.join(OUT_DIR, f)) / 1024)
                                                            for f in new]}),
                     "Present from your own screen; do not share the files."])